Dans ce notebook on réalise le preprocessing de la base de données à partir des conclusions tirées de l'EDA. Le split La variable duration a été exclue pour éviter toute fuite d’information. Les pseudo-manquants ont été conservés comme modalités informatives. Les variables fortement asymétriques ont été transformées de manière robuste, et la variable pdays a fait l’objet d’un recodage spécifique afin de capturer l’historique de contact sans explosion de dimension. Les variables catégorielles ont été encodées par one-hot encoding.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, pointbiserialr
from itertools import combinations
df = pd.read_csv("bank-full.csv", sep=';')

# 1. Nettoyage conceptuel : 
## Variables à exclure : duration (fuite d’information (connue après l’appel))

In [14]:
df = df.drop(columns=["duration"])

PHASE 1 — Cible et split

Encodage de la cible

In [15]:
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y"])

Split stratifié : 

In [16]:
from sklearn.model_selection import StratifiedShuffleSplit

# split stratifié sur la cible
stratified = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

# IMPORTANT: utiliser X et y, et .iloc (pas .loc)
for train_idx, test_idx in stratified.split(X, y):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test  = y.iloc[test_idx].copy()

PHASE 3 — Traitement des pseudo-manquants
On NE supprime PAS les modalités unknown

PHASE 4 — Cas spécial : pdays
Constat EDA
pdays = -1 = jamais contacté (81%)
Très forte cardinalité
Corrélé à previous
Décision de preprocessing (propre et défendable)
👉 Créer 2 variables à partir de pdays :
Indicateur binaire : déjà contacté ?
Version tronquée / binée pour les valeurs positives

In [ ]:
def preprocess_pdays(df):
    df = df.copy()
    df["pdays_contacted"] = (df["pdays"] != -1).astype(int)
    df["pdays_positive"] = df["pdays"].clip(lower=0)
    df = df.drop(columns=["pdays"])
    return df
X_train = preprocess_pdays(X_train)
X_test  = preprocess_pdays(X_test)

In [22]:
print("pdays_contacted unique:", sorted(X_train["pdays_contacted"].unique()))
print("pdays_positive min:", X_train["pdays_positive"].min())

assert set(X_train["pdays_contacted"].unique()).issubset({0, 1})
assert X_train["pdays_positive"].min() >= 0


pdays_contacted unique: [0, 1]
pdays_positive min: 0


PHASE 5 — Variables très asymétriques (numériques)
Constat EDA
balance, campaign, previous → longues queues
Pas des erreurs → hétérogénéité réelle
Décisions
transformation log1p 

La variable balance présentant des valeurs positives et négatives fortement asymétriques, une transformation logarithmique signée de type sign(x)·log(1+|x|) a été utilisée afin de stabiliser la variance tout en conservant l’information de signe.

In [ ]:
def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

skewed_vars = ["balance", "campaign", "previous"]

for col in skewed_vars:
    X_train[col] = signed_log1p(X_train[col])
    X_test[col]  = signed_log1p(X_test[col])

In [ ]:
skewed_vars = ["balance", "campaign", "previous"]

print("NaN counts:\n", X_train[skewed_vars].isna().sum())
print("Inf counts:\n", np.isinf(X_train[skewed_vars]).sum())

assert X_train[skewed_vars].isna().sum().sum() == 0
assert np.isinf(X_train[skewed_vars].to_numpy()).sum() == 0


In [ ]:
# comparaison signe avant/après : si tu as encore le df original quelque part
# sinon check simple: balance transformée doit avoir des + et des - si balance brute en avait.
print("balance transformed min/max:", X_train["balance"].min(), X_train["balance"].max())


PHASE 6 — Encodage des variables catégorielles
Beaucoup de variables catégorielles
Signal principalement porté par elles
Cardinalité modérée (sauf job)
Décision RECOMMANDÉE (StatApp-safe)
👉 One-Hot Encoding
handle_unknown="ignore"
éventuellement min_frequency pour job
permet de capturer des effets non linéaires et interactions implicites,

In [ ]:
cols = ["job", "marital", "education", "contact", "month", "poutcome"]
for c in cols:
    print(c, X_train[c].nunique(), sorted(X_train[c].unique())[:10])

In [19]:
# Encodage des variables
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer


categorical_nominal = ["job", "marital", "education", "contact", "month", "poutcome"]
categorical_binary  = ["default", "housing", "loan"]

numerical_features = [
    "age", "balance", "day",
    "campaign", "previous",
    "pdays_contacted", "pdays_positive"
]

#making preprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat_nom", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_nominal),
        ("cat_bin", OneHotEncoder(drop="if_binary"), categorical_binary)
    ]
)

PHASE 7 — assemblage final 

In [20]:
X_train_final = preprocessor.fit_transform(X_train)
X_test_final  = preprocessor.transform(X_test)

In [21]:
print("Proportion y=1 train:", y_train.mean())
print("Proportion y=1 test :", y_test.mean())
print("Shapes:", X_train_final.shape, X_test_final.shape)

Proportion y=1 train: 0.11698186241981863
Proportion y=1 test : 0.11699657193409267
Shapes: (36168, 42) (9043, 42)


In [24]:
feature_names = preprocessor.get_feature_names_out()

print("Nombre de features :", len(feature_names))
feature_names

Nombre de features : 42


array(['num__age', 'num__balance', 'num__day', 'num__campaign',
       'num__previous', 'num__pdays_contacted', 'num__pdays_positive',
       'cat_nom__job_blue-collar', 'cat_nom__job_entrepreneur',
       'cat_nom__job_housemaid', 'cat_nom__job_management',
       'cat_nom__job_retired', 'cat_nom__job_self-employed',
       'cat_nom__job_services', 'cat_nom__job_student',
       'cat_nom__job_technician', 'cat_nom__job_unemployed',
       'cat_nom__job_unknown', 'cat_nom__marital_married',
       'cat_nom__marital_single', 'cat_nom__education_secondary',
       'cat_nom__education_tertiary', 'cat_nom__education_unknown',
       'cat_nom__contact_telephone', 'cat_nom__contact_unknown',
       'cat_nom__month_aug', 'cat_nom__month_dec', 'cat_nom__month_feb',
       'cat_nom__month_jan', 'cat_nom__month_jul', 'cat_nom__month_jun',
       'cat_nom__month_mar', 'cat_nom__month_may', 'cat_nom__month_nov',
       'cat_nom__month_oct', 'cat_nom__month_sep',
       'cat_nom__poutcome_other', '